In [71]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/"

import os
os.chdir(r"C:\Z")  # Cambia el directorio de trabajo
print(os.getcwd())  # Verifica que cambió correctamente el directorio base

C:\Z


In [72]:
from SciServer import CasJobs as cj
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
pd.set_option('display.max_rows', 20)

# Iniciar sesión
from SciServer import Authentication
Authentication.login(UserName = "Alberto_2002", Password = "#Blackie2002")

'1b5424fdd1c9444abfab1b70e33b76df'

In [73]:
# import os
# import requests
# from SciServer import CasJobs

# # Seleccionar la URL a la que está asociada el archivo fits
# sql_query = """
# SELECT TOP 15000
#     p.objid,
#     p.ra,
#     p.dec,
#     s.specObjID,
#     dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
#     s.z AS redshift
# FROM 
#     PhotoObj AS p
# JOIN 
#     SpecObj AS s ON s.bestobjid = p.objid
# WHERE 
#     s.z BETWEEN 0.3 AND 0.7
#     AND s.zWarning = 0
# """

# sql_query = """
# SELECT TOP 1000000
#     p.objid,
#     p.ra,
#     p.dec,
#     s.specObjID,
#     dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
#     s.z AS redshift
# FROM 
#     PhotoObj AS p
# JOIN 
#     SpecObj AS s ON s.bestobjid = p.objid
# WHERE 
#     s.z BETWEEN 0.3 AND 0.7
#     AND s.zWarning = 0
# """

# results = CasJobs.executeQuery(sql_query, context="DR18")
# fits_urls = results["fits_url"].tolist()

In [74]:
import os
import requests
import time
import pandas as pd
from SciServer import CasJobs

# Parámetros iniciales
batch_size_inicial = 1000       # tamaño inicial del lote
min_batch_size = 1000           # tamaño mínimo de lote permitido
max_retries = 10                # máximo número de reintentos por lote
offset = 0                      # registro inicial a recuperar
all_results = []
batch_results = []

while offset <= 1000000:
  
    current_batch_size = batch_size_inicial
    retries = 0
    success = False
    iteration_start = time.time()  # marca de tiempo para controlar el rate limit

    # Intentar ejecutar el query con el tamaño de lote actual. Si hay error, se reduce el lote y se reintenta.
    while not success and retries < max_retries:
        sql_query = f"""
        SELECT
            p.objid,
            s.specObjID,
            dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
            s.z AS redshift
        FROM 
            PhotoObj AS p
        JOIN 
            SpecObj AS s ON s.bestobjid = p.objid
        WHERE 
            s.z BETWEEN 0 AND 1
            AND s.zWarning = 0
            AND (s.class = 'QSO' OR s.class = 'GALAXY')
        ORDER BY s.z
        OFFSET {offset} ROWS
        FETCH NEXT {current_batch_size} ROWS ONLY
        """
        try:
            batch_results = pd.DataFrame(CasJobs.executeQuery(sql_query, context="DR18"))
            success = True
            all_results.append(batch_results)
            offset += len(batch_results)
            print(f"Recuperados {offset} registros hasta ahora.")

        except Exception as e:
            print(f"Error en la consulta con batch_size = {current_batch_size}: {e}")
            # Reducir el tamaño del lote, pero sin bajar de min_batch_size
            current_batch_size = max(min_batch_size, current_batch_size // 2)
            retries += 1
            time.sleep(2)  # Esperar un poco antes de reintentar

    if len(batch_results) == 0:
        print("No hay más registros para recuperar.")
        ultimo_redshift = all_results[len(all_results)-2]["redshift"].iloc[0]
        print(f"Último redshift recuperado: {ultimo_redshift}")
        break

    if retries == max_retries:
        print("No se pudo recuperar el batch tras varios reintentos. Terminando.")
        ultimo_redshift = all_results[len(all_results)-2]["redshift"].iloc[0]
        print(f"Último redshift recuperado: {ultimo_redshift}")
        break

Recuperados 1000 registros hasta ahora.
Recuperados 2000 registros hasta ahora.
Recuperados 2291 registros hasta ahora.
Recuperados 2291 registros hasta ahora.
No hay más registros para recuperar.
Último redshift recuperado: 0.199879


In [75]:
# import os
# from astropy.io import fits
# import numpy as np

# # Ruta a la carpeta donde se encuentran los archivos FITS
# folder_path = r'TFGF NO DRIVE/Python/spectrums'

# bestobjid_list = []
# i = 0

# # Recorrer todos los archivos en la carpeta
# for filename in os.listdir(folder_path):
#     if filename.endswith('.fits'):
#         i += 1
#         print(f"Procesando archivo {i}")
#         file_path = os.path.join(folder_path, filename)
#         with fits.open(file_path) as hdul:
#             try:
#                 # Se asume que la columna BESTOBJID se encuentra en hdul[2]
#                 bestobjid_data = hdul[2].data['BESTOBJID']
#                 # Iterar sobre los elementos y convertir cada uno a entero
#                 for value in bestobjid_data:
#                     try:
#                         int_value = int(value)
#                         bestobjid_list.append(int_value)
#                     except Exception as e:
#                         print(f"No se pudo convertir {value} a entero: {e}")
#             except Exception as e:
#                 print(f"Error en el archivo {filename}: {e}")

# if bestobjid_list:
#     max_objid = max(bestobjid_list)
#     print("El mayor BESTOBJID encontrado es:", max_objid)
# else:
#     print("No se encontraron valores de BESTOBJID en los archivos.")

In [76]:
# import os
# import requests
# import time
# import pandas as pd
# from SciServer import CasJobs as cj

# last_objid = 0  # o el valor inicial apropiado

# all_results = []
# batch_results = []



# while True:
#     sql_query = f"""
#     SELECT
#         p.objid,
#         s.specObjID,
#         dbo.fGetUrlFitsSpectrum(s.specObjID) AS fits_url,
#         s.z AS redshift
#     FROM PhotoObj AS p
#     JOIN SpecObj AS s ON s.bestobjid = p.objid
#     WHERE s.zWarning = 0
#       AND p.objid > {last_objid}
#     ORDER BY p.objid
#     """
#     try:
#         batch_results = pd.DataFrame(cj.executeQuery(sql_query, context="DR18"))
#         batch_results = []
#         if batch_results.empty:
#             break  # No hay más registros.
#         all_results.append(batch_results)
#         last_objid = batch_results['objid'].max()  # Actualizamos el último valor
#         print(f"Recuperados hasta objid {last_objid}.")

#     except Exception as e:
#         print(f"Error al recuperar el batch: {e}")
#         break

In [77]:
# Combinar todos los resultados (suponiendo que cada batch es un DataFrame)
combined_results = pd.concat(all_results, ignore_index=True)
print(f"Total de registros recuperados: {len(combined_results)}")

# Extraer las URLs de los archivos FITS
fits_urls = combined_results["fits_url"].tolist()
print(f"Se han obtenido {len(fits_urls)} URLs de FITS.")

Total de registros recuperados: 2291
Se han obtenido 2291 URLs de FITS.


C:\Users\alfa\AppData\Local\Temp\ipykernel_17768\4134409215.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_results = pd.concat(all_results, ignore_index=True)


In [78]:
import os
import requests
import time
import pandas as pd
from SciServer import CasJobs

# Directorio de descarga
output_dir = r"C:/Z/TFGF NO DRIVE/Python/spectrums"
os.makedirs(output_dir, exist_ok=True)

archivos_descargados = []

def download_file(url):
    filename = os.path.basename(url)
    local_path = os.path.join(output_dir, filename)
    
    # Saltar si ya se descargó este archivo
    if os.path.exists(local_path):
        print(f"Saltando {filename} (ya descargado).")
        return filename, None
    
    try:
        response = requests.get(url)
        response.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(response.content)
        print(f"Descargado: {filename}")
        return filename, None
    except Exception as e:
        print(f"Error al descargar {url}: {e}")
        return filename, e

# Número de descargas en paralelo
paralelo = 100
i=0

with ThreadPoolExecutor(max_workers=paralelo) as executor:
    future_to_url = {executor.submit(download_file, url): url for url in fits_urls}
    
    for future in as_completed(future_to_url):
        i+=1
        print(f"Descargando archivo {i} de {len(fits_urls)}")
        filename, error = future.result()
        if error is None:
            archivos_descargados.append(filename)

print("Descarga finalizada")

Saltando spec-7695-57654-0131.fits (ya descargado).Saltando spec-0837-52642-0348.fits (ya descargado).

Saltando spec-1589-52972-0309.fits (ya descargado).
Saltando spec-0813-52354-0292.fits (ya descargado).
Saltando spec-5054-56191-0042.fits (ya descargado).
Saltando spec-0566-52238-0171.fits (ya descargado).
Saltando spec-3855-55268-0314.fits (ya descargado).
Saltando spec-0457-51901-0423.fits (ya descargado).
Saltando spec-0827-52312-0431.fits (ya descargado).
Saltando spec-1351-52790-0513.fits (ya descargado).
Saltando spec-1243-52930-0449.fits (ya descargado).
Saltando spec-6123-56217-0104.fits (ya descargado).
Saltando spec-0978-52441-0240.fits (ya descargado).
Saltando spec-4550-55894-0753.fits (ya descargado).
Saltando spec-0886-52381-0570.fits (ya descargado).
Saltando spec-7731-58130-0927.fits (ya descargado).
Saltando spec-7837-56987-0046.fits (ya descargado).
Saltando spec-0545-52202-0193.fits (ya descargado).
Saltando spec-11633-58463-0112.fits (ya descargado).
Saltando sp